# LangGraph fundamentals

Assistant version 9: Classify, check policy, interrupt for approval, record refund and write a local notification outbox.

Read `README.md` for the analogies, diagrams, API explanations, common mistakes and quiz. These cells contain the same source as the scripts; edit inputs and rerun a cell to explore. Outputs are intentionally cleared in Git. This notebook executes scripted fixtures with network blocked, not real models. Use the script commands without `--offline` for live experiments.

Start with `uv sync --extra deep --group authoring`, then `uv run --extra deep --group authoring jupyter lab` from the repository root.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ["COURSE_OFFLINE"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"
from common.testing import offline_environment

print("OFFLINE FIXTURES: no API fees; outputs do not measure model quality.")

## StateGraph; TypedDict and Pydantic state; nodes, edges, START, END; compile

Predict the output before running. The function definitions below match `examples/01_state_graph.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Compare two state schemas for the same return check.

Run: uv run --extra agents python modules/14_langgraph/examples/01_state_graph.py --offline
Expected: Both graphs route a 12,195-cent refund to review.
Cost: No model calls. Offline fixtures incur no API fee.
Requires: agents
"""

from typing import TypedDict

from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


class ReturnState(TypedDict):
    cents: int
    review: bool


class ValidatedState(BaseModel):
    cents: int = Field(ge=0)
    review: bool = False


def check_dict(state: ReturnState) -> dict:
    return {"review": state["cents"] > 10000}


def check_model(state: ValidatedState) -> dict:
    return {"review": state.cents > 10000}


def main() -> None:
    lesson_settings()
    for schema, node in [(ReturnState, check_dict), (ValidatedState, check_model)]:
        # KEY IDEA: a node returns a partial update, not a mutated shared dictionary.
        builder = StateGraph(schema)
        builder.add_node("check", node)
        builder.add_edge(START, "check")
        builder.add_edge("check", END)
        graph = builder.compile()
        print_text(str(graph.invoke({"cents": 12195, "review": False})))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## add_messages and operator.add reducers; concurrent writes

Predict the output before running. The function definitions below match `examples/02_reducers.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Merge concurrent evidence and replace a corrected message by ID.

Run: uv run --extra agents python modules/14_langgraph/examples/02_reducers.py --offline
Expected: Two evidence records; one corrected message.
Cost: No model calls. Offline fixtures incur no API fee.
Requires: agents
"""

import operator
from typing import Annotated, TypedDict

from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.errors import InvalidUpdateError
from langgraph.graph import END, START, StateGraph, add_messages

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


class State(TypedDict):
    evidence: Annotated[list[str], operator.add]
    messages: Annotated[list[AnyMessage], add_messages]


class Unreduced(TypedDict):
    evidence: list[str]


def stock(state: State) -> dict:
    return {"evidence": ["BW-001 stock requires a fresh lookup"]}


def policy(state: State) -> dict:
    return {"evidence": ["Unused returns have a 30-day window"]}


def main() -> None:
    lesson_settings()
    broken = StateGraph(Unreduced)
    for name, node in [("stock", stock), ("policy", policy)]:
        broken.add_node(name, node, input_schema=Unreduced)
        broken.add_edge(START, name)
        broken.add_edge(name, END)
    try:
        broken.compile().invoke({"evidence": []})
    except InvalidUpdateError:
        print_text("Expected InvalidUpdateError: parallel evidence needs a reducer.")
    else:
        raise AssertionError("The unreduced parallel graph should reject concurrent writes.")
    graph = StateGraph(State)
    for name, node in [("stock", stock), ("policy", policy)]:
        graph.add_node(name, node)
        graph.add_edge(START, name)
        graph.add_edge(name, END)
    # KEY IDEA: list reducers collect parallel updates; sort when order matters.
    result = graph.compile().invoke({"evidence": [], "messages": []})
    messages = add_messages(
        [HumanMessage("Wrong order", id="request")], [HumanMessage("Order BW1001", id="request")]
    )
    print_text(str({"evidence": sorted(result["evidence"]), "messages": messages}))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Conditional edges; Command update and goto

Predict the output before running. The function definitions below match `examples/03_routing_command.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Route once with a conditional edge, then update and route with Command.

Run: uv run --extra agents python modules/14_langgraph/examples/03_routing_command.py --offline
Expected: The refund takes the review path; status is review requested.
Cost: No model calls. Offline fixtures incur no API fee.
Requires: agents
"""

from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Command

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


class State(TypedDict):
    cents: int
    status: str


def route(state: State) -> Literal["review", "finish"]:
    return "review" if state["cents"] > 10000 else "finish"


def review(state: State) -> Command[Literal["finish"]]:
    return Command(update={"status": "review requested"}, goto="finish")


def main() -> None:
    lesson_settings()
    graph = StateGraph(State)
    graph.add_node("review", review)
    graph.add_node("finish", lambda state: {})
    graph.add_conditional_edges(START, route)
    # KEY IDEA: Command is the review node's only outgoing route; no duplicate edge.
    graph.add_edge("finish", END)
    print_text(str(graph.compile().invoke({"cents": 12195, "status": "received"})))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Build it yourself: model node, ToolNode, conditional edge and bounded loop

Predict the output before running. The function definitions below match `examples/04_manual_agent_graph.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Build the model-tool loop by hand using ToolNode.

Run: uv run --extra agents python modules/14_langgraph/examples/04_manual_agent_graph.py --offline
Expected: One stock tool result and a final model answer.
Cost: At most three short model calls; a tool-calling model is required. Offline fixtures incur
no API fee.
Requires: agents
"""

from typing import Annotated, Literal, TypedDict

from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph import END, START, StateGraph, add_messages
from langgraph.prebuilt import ToolNode

from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_message
from common.tools import make_order_tools


class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    calls: int


def main() -> None:
    settings = lesson_settings()
    tools = [make_order_tools("C001")[2]]
    model = get_chat_model(
        settings=settings,
        responses=[tool_request("check_stock", {"sku": "BW-001"}), "Stock checked."],
    ).bind_tools(tools)

    def call(state: State) -> dict:
        return {"messages": [model.invoke(state["messages"])], "calls": state["calls"] + 1}

    def route(state: State) -> Literal["tools", "__end__"]:
        if state["messages"][-1].tool_calls:
            if state["calls"] >= 3:
                raise ValueError("Three-call budget exhausted before another tool dispatch.")
            return "tools"
        return END

    # KEY IDEA: the loop is ordinary graph routing with an explicit call ceiling.
    graph = StateGraph(State)
    graph.add_node("model", call)
    graph.add_node("tools", ToolNode(tools))
    graph.add_edge(START, "model")
    graph.add_conditional_edges("model", route)
    graph.add_edge("tools", "model")
    result = graph.compile().invoke(
        {"messages": [HumanMessage("Check BW-001 stock")], "calls": 0}, {"recursion_limit": 10}
    )
    for message in result["messages"]:
        print_message(message)


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Checkpointers; durable execution; idempotent side effects

Predict the output before running. The function definitions below match `examples/05_persistence.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Reopen a SQLite workflow and replay an idempotent return.

Run: uv run --extra agents python modules/14_langgraph/examples/05_persistence.py --offline
Expected: One return and one outbox row after repeated requests.
Cost: No model calls. Offline fixtures incur no API fee.
Requires: agents
"""

import sqlite3
from contextlib import closing
from pathlib import Path
from tempfile import TemporaryDirectory

from langgraph.checkpoint.sqlite import SqliteSaver

from common.errors import run_cli
from common.graphs import build_return_graph
from common.lesson import lesson_settings
from common.memory import thread_config
from common.pretty import print_text
from common.returns import SupportContext, runtime_orders


def main() -> None:
    lesson_settings()
    with TemporaryDirectory() as folder:
        context = SupportContext(
            "C008",
            runtime_orders(Path(folder) / "orders.db"),
            inspected_unused=True,
            proof_and_accessories=True,
        )
        request = {
            "order_id": "BW1008",
            "sku": "BW-021",
            "reason": "unused",
            "idempotency_key": "persistent-return",
        }
        config = thread_config("C008", "return")
        # KEY IDEA: durable state is separate from idempotency in the business database.
        for _ in range(2):
            with SqliteSaver.from_conn_string(str(Path(folder) / "graph.db")) as saver:
                graph = build_return_graph(saver)
                result = graph.invoke({"request": request}, config, context=context)
                print_text(result["status"])
        with closing(sqlite3.connect(context.database)) as db:
            print_text(str(db.execute("SELECT COUNT(*) FROM notification_outbox").fetchone()))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## interrupt; Command(resume=...); approval and rejection branches

Predict the output before running. The function definitions below match `examples/06_interrupt.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Pause before a side effect and resume a human decision.

Run: uv run --extra agents python modules/14_langgraph/examples/06_interrupt.py --offline
Expected: Approve and reject take different branches; neither writes before review.
Cost: No model calls. Offline fixtures incur no API fee.
Requires: agents
"""

from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

from common.errors import run_cli
from common.lesson import lesson_settings
from common.memory import thread_config
from common.pretty import print_text


class State(TypedDict):
    request: str
    decision: str


def review(state: State) -> dict:
    answer = interrupt({"request": state["request"], "choices": ["approve", "reject"]})
    if answer not in ("approve", "reject"):
        raise ValueError("Invalid review decision")
    return {"decision": answer}


def main() -> None:
    lesson_settings()
    builder = StateGraph(State)
    builder.add_node("review", review)
    builder.add_edge(START, "review")
    builder.add_edge("review", END)
    graph = builder.compile(checkpointer=InMemorySaver())
    for decision in ("approve", "reject"):
        config = thread_config("C001", decision)
        pause = graph.invoke({"request": "Review $121.95 refund", "decision": "pending"}, config)
        print_text(str(pause["__interrupt__"]))
        # KEY IDEA: only a trusted reviewer interface should construct the resume command.
        print_text(str(graph.invoke(Command(resume=decision), config)))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## State history; replay; fork from checkpoint

Predict the output before running. The function definitions below match `examples/07_time_travel.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Inspect history, replay a checkpoint, and fork a read-only workflow.

Run: uv run --extra agents python modules/14_langgraph/examples/07_time_travel.py --offline
Expected: Original standard quote stays available; fork uses expedited.
Cost: No model calls. Offline fixtures incur no API fee.
Requires: agents
"""

from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph

from common.errors import run_cli
from common.lesson import lesson_settings
from common.memory import thread_config
from common.pretty import print_text


class State(TypedDict):
    shipping: str
    quote: str


def quote(state: State) -> dict:
    return {
        "quote": "3-5 business days" if state["shipping"] == "standard" else "1-2 business days"
    }


def main() -> None:
    lesson_settings()
    builder = StateGraph(State)
    builder.add_node("quote", quote)
    builder.add_edge(START, "quote")
    builder.add_edge("quote", END)
    graph = builder.compile(checkpointer=InMemorySaver())
    config = thread_config("C001", "shipping")
    graph.invoke({"shipping": "standard", "quote": ""}, config)
    history = list(graph.get_state_history(config))
    before = next(snapshot for snapshot in history if snapshot.next == ("quote",))
    replay = graph.invoke(None, before.config)
    # KEY IDEA: update_state returns the fork config; it does not rewrite historical snapshots.
    fork = graph.update_state(before.config, {"shipping": "expedited"}, as_node="__start__")
    changed = graph.invoke(None, fork)
    print_text(
        str(
            {
                "replay": replay,
                "fork": changed,
                "original": graph.get_state(history[0].config).values,
            }
        )
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Subgraphs; private/shared state; recursion limits

Predict the output before running. The function definitions below match `examples/08_subgraphs.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Map private child state into the parent and show a recursion backstop.

Run: uv run --extra agents python modules/14_langgraph/examples/08_subgraphs.py --offline
Expected: Private notes stay inside the child; an intentional loop hits its limit.
Cost: No model calls. Offline fixtures incur no API fee.
Requires: agents
"""

from typing import TypedDict

from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


class Parent(TypedDict):
    question: str
    answer: str


class Child(TypedDict):
    query: str
    private_note: str
    answer: str


class Shared(TypedDict):
    query: str
    answer: str


def inspect(state: Child) -> dict:
    return {"private_note": "internal routing detail", "answer": "Unused returns: 30 days."}


def main() -> None:
    lesson_settings()
    child = StateGraph(Child)
    child.add_node("inspect", inspect)
    child.add_edge(START, "inspect")
    child.add_edge("inspect", END)
    compiled = child.compile()
    shared = StateGraph(Shared)
    shared.add_node("child", compiled)
    shared.add_edge(START, "child")
    shared.add_edge("child", END)
    shared_result = shared.compile().invoke({"query": "Return window?", "answer": ""})
    assert "private_note" not in shared_result
    print_text(str(shared_result))

    def delegate(state: Parent) -> dict:
        result = compiled.invoke({"query": state["question"], "private_note": "", "answer": ""})
        return {"answer": result["answer"]}

    # KEY IDEA: explicit mapping is a context boundary; shared keys alone are not isolation.
    parent = StateGraph(Parent)
    parent.add_node("delegate", delegate)
    parent.add_edge(START, "delegate")
    parent.add_edge("delegate", END)
    print_text(str(parent.compile().invoke({"question": "Return window?", "answer": ""})))
    loop = StateGraph(Parent)
    loop.add_node("again", lambda state: {})
    loop.add_edge(START, "again")
    loop.add_edge("again", "again")
    try:
        loop.compile().invoke({"question": "loop", "answer": ""}, {"recursion_limit": 3})
    except GraphRecursionError:
        print_text("Expected GraphRecursionError: add a termination branch.")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## values, updates, messages, custom streams; progress events

Predict the output before running. The function definitions below match `examples/09_stream_modes.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Inspect snapshots, deltas, model tokens and custom progress.

Run: uv run --extra agents python modules/14_langgraph/examples/09_stream_modes.py --offline
Expected: All four stream modes emit distinct kinds of events.
Cost: One short model call. Offline fixtures incur no API fee.
Requires: agents
"""

from typing import TypedDict

from langgraph.config import get_stream_writer
from langgraph.graph import END, START, StateGraph

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text


class State(TypedDict):
    question: str
    answer: str


def main() -> None:
    settings = lesson_settings()
    model = get_chat_model(settings=settings)

    def draft(state: State) -> dict:
        get_stream_writer()({"stage": "drafting", "progress": 0})
        reply = model.invoke(state["question"])
        get_stream_writer()({"stage": "complete", "progress": 100})
        return {"answer": str(reply.content)}

    builder = StateGraph(State)
    builder.add_node("draft", draft)
    builder.add_edge(START, "draft")
    builder.add_edge("draft", END)
    # KEY IDEA: a mode list labels each event; progress is not model-generated text.
    for mode, event in builder.compile().stream(
        {"question": "Explain 30-day returns.", "answer": ""},
        stream_mode=["values", "updates", "messages", "custom"],
    ):
        print_text(f"{mode}: {event}")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## @entrypoint; @task; task-result replay; sync and async entry points

Predict the output before running. The function definitions below match `examples/10_functional_api.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Reuse a persisted task result after interrupt, then run an async entrypoint.

Run: uv run --extra agents python modules/14_langgraph/examples/10_functional_api.py --offline
Expected: The drafting task executes once across pause/resume; async workflow returns.
Cost: No model calls. Offline fixtures incur no API fee.
Requires: agents
"""

import asyncio

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.func import entrypoint, task
from langgraph.types import Command, interrupt

from common.errors import run_cli
from common.lesson import lesson_settings
from common.memory import thread_config
from common.pretty import print_text


def main() -> None:
    lesson_settings()
    calls = []

    @task
    def draft(topic: str) -> str:
        calls.append(topic)
        return "Brightwave draft: " + topic

    @entrypoint(checkpointer=InMemorySaver())
    def workflow(topic: str) -> dict:
        text = draft(topic).result()
        approved = interrupt({"draft": text})
        if type(approved) is not bool:
            raise ValueError("Review must be boolean")
        return {"draft": text, "approved": approved}

    config = thread_config("C001", "functional")
    workflow.invoke("repairable gear", config)
    # KEY IDEA: entrypoint code restarts, but the completed task result is replayed.
    print_text(str(workflow.invoke(Command(resume=False), config)))
    print_text(f"Draft task executions: {len(calls)}")

    @task
    async def read_topic(topic: str) -> str:
        return "Synthetic research: " + topic

    @entrypoint()
    async def async_workflow(topic: str) -> str:
        return await read_topic(topic)

    print_text(asyncio.run(async_workflow.ainvoke("tent setup")))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## draw_mermaid; local LangGraph CLI dev server; Studio; account-free diagram alternative

Predict the output before running. The function definitions below match `examples/11_studio.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Draw an account-free diagram and print the optional local-server command.

Run: uv run --extra agents python modules/14_langgraph/examples/11_studio.py --offline
Expected: Mermaid graph text; the example does not start a persistent service.
Cost: No model calls. Offline fixtures incur no API fee.
Requires: agents
"""

from common.errors import run_cli
from common.graphs import studio_graph
from common.lesson import lesson_settings
from common.pretty import print_text


def main() -> None:
    lesson_settings()
    # KEY IDEA: draw_mermaid produces text locally; no rendering service is needed.
    print_text(studio_graph.get_graph().draw_mermaid())
    print_text(
        "Optional: uv run --extra deployment langgraph dev --config "
        "langgraph.json --no-browser --host 127.0.0.1"
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world use case: Classify, check policy, interrupt for approval, record refund and write a local notification outbox

Predict the output before running. The function definitions below match `use_case/return_workflow.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Record a reviewed synthetic return and queue one local notification.

Run: uv run --extra agents python modules/14_langgraph/use_case/return_workflow.py --offline
Expected: Pause with no write, then an approved record and one queued notice.
Cost: No model or delivery service calls. Requires: agents
"""

from pathlib import Path
from tempfile import TemporaryDirectory

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command

from common.errors import run_cli
from common.graphs import build_return_graph
from common.lesson import lesson_settings
from common.memory import thread_config
from common.pretty import print_text
from common.returns import SupportContext, runtime_orders


def main() -> None:
    lesson_settings()
    with TemporaryDirectory() as folder:
        context = SupportContext(
            "C001",
            runtime_orders(Path(folder) / "orders.db"),
            inspected_unused=True,
            proof_and_accessories=True,
        )
        config = thread_config("C001", "return")
        with SqliteSaver.from_conn_string(str(Path(folder) / "graph.db")) as saver:
            graph = build_return_graph(saver)
            state = graph.invoke(
                {
                    "request": {
                        "order_id": "BW1001",
                        "sku": "BW-014",
                        "reason": "unused",
                        "idempotency_key": "workflow-demo",
                    }
                },
                config,
                context=context,
            )
            print_text(str(state["__interrupt__"]))
            # KEY IDEA: this is a simulated reviewer, never a model-generated approval.
            result = graph.invoke(Command(resume="approve"), config, context=context)
            print_text(str(result))
            print_text("Notification queued locally, not sent. Temporary demo data is discarded.")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Practice and reflect

Change a Brightwave input and rerun the relevant cell. Inspect whether the prompt, tool arguments or parser result changed. A scripted model reply may stay the same: that is a fixture limitation, not evidence that a real model ignores input. Complete the three exercises in `exercises/`, run their target checks, and answer the quiz in `quiz.md`. Leave outputs cleared before committing.

Run `uv run --extra agents pytest modules/14_langgraph/tests` for the supplied solution checks.